# Tracking Lunar Gateway with an uncontrolled halo observer

The uncontrolled halo observer is used to track Lunar Gateway using optical angle measurements. Three lunar surface observatories (LSOs) estimate the observer's orbit, while the observer measures Gateway. The analysis compares Gateway tracking with perfect and estimated knowledge of the observer's state, then examines how tracking degrades when the observer departs from the nominal halo and Gateway measurements become unavailable.

The long-term goal is to maintain **Gateway position-estimation error below 100 m** and **velocity-estimation error below 0.01 m/s**. The halo observer can tolerate a much larger departure from its nominal orbit as long as Gateway tracking remains accurate. The figures below establish the uncontrolled baseline. Later sections generate executed-burn trajectories for training and testing a nonlinear ARX guidance model.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the project checkout")
sys.path.insert(0, str(ROOT / "src"))
from halo_tracking.study import load_config
cfg = load_config(ROOT / "configs/baseline.json")
cfg["simulation"]["periods"] = 1.0  # One nominal observer period


## 1. Matched comparison

| Assumption | Perfect observer | LSO-estimated observer |
|---|---|---|
| Observer state supplied to Gateway estimator | Exact true state | State estimated from LSO angles |
| Initial observer uncertainty | Zero | 1 km and 0.1 m/s per axis, 1σ |
| Initial Gateway uncertainty | 1 km and 0.1 m/s per axis, 1σ | Same |
| Observer-to-Gateway camera | 1 arcsecond per angle | Same measurements and noise |
| LSO | Not needed | 30° N at 270° E and 90° E, 85° S at 90° E, 1 arcsecond per angle |

These are the baseline settings. The configuration table below displays the values actually used. Both optical links use **angles only** in this focused comparison. The cadence is 30 minutes. Earth/Moon occultation is included for Gateway observations. The LSO uses the configured elevation mask. Sun 20° / Earth 10° keep-outs and the configured range cap apply to both links. Optical brightness and camera attitude errors are not modeled.

The true observer remains on the supplied initial-value trajectory in both cases. The imperfect case introduces uncertainty in our **knowledge** of it, not a different physical orbit.

In [ ]:
from IPython.display import HTML, Markdown, display
settings = [
    ("Duration", f"1 observer nominal period = {cfg['orbit']['period_tu'] * cfg['time_unit_s'] / 86400:.2f} days"),
    ("Observer initial position / velocity 1σ", f"{cfg['estimation']['initial_position_sigma_km']:g} km / {1000*cfg['estimation']['initial_velocity_sigma_km_s']:g} m/s per axis"),
    ("Gateway initial position / velocity 1σ", f"{cfg['comparison']['gateway_position_sigma_km']:g} km / {1000*cfg['comparison']['gateway_velocity_sigma_km_s']:g} m/s per axis"),
    ("LSO angular noise 1σ", f"{cfg['measurement']['angle_sigma_arcsec']:g} arcsec per angle"),
    ("Gateway camera angular noise 1σ", f"{cfg['comparison']['gateway_camera_sigma_arcsec']:g} arcsec per angle")]
display(HTML("<table>"+"".join(f"<tr><th>{name}</th><td>{value}</td></tr>" for name,value in settings)+"</table>"))

In [ ]:
from halo_tracking.gateway_comparison import run_gateway_comparison, save_comparison
result = run_gateway_comparison(cfg)
report = save_comparison(result, ROOT / "results/gateway_comparison")

## 2. Three lunar observatories estimate the halo observer

The three LSOs observe the halo spacecraft from different locations on the Moon. Their angular measurements allow the observer's orbit to be estimated even though its initial state is uncertain. The geometry plot shows the Moon, observer and Gateway trajectories. The following plot compares the estimated observer orbit with its true simulated trajectory.

The observer's orbit is the same in the perfect-knowledge and estimated-knowledge cases. Only the quality of the information available to the Gateway tracking filter changes. The network uses three stations to improve observation coverage; the current experiment does not establish that three stations are the minimum required.

The optical measurement is an azimuth/elevation pair:

$$
\mathbf{z}_{k}=h(\mathbf{x}_{k})+\boldsymbol{\eta}_{k},\qquad
\boldsymbol{\eta}_{k}\sim\mathcal{N}(\mathbf{0},R).
$$

Only measurements that pass the elevation, occultation and keep-out checks update the observer estimate. The estimated uncertainty is obtained from the filter covariance, while the actual error is measured against simulated truth.

In [ ]:
from halo_tracking.plotting import geometry, observer_knowledge, gateway_comparison_figures, state_histories, export
fig_geometry = geometry(result)
export(fig_geometry, ROOT / "results/gateway_comparison/geometry.png")

In [ ]:
fig_observer = observer_knowledge(result)
export(fig_observer, ROOT / "results/gateway_comparison/observer.png")

## Observer position, velocity and acceleration

The solid curve is the perfect observer state (truth). The dashed curve is the LSO-predicted state. Components use Moon-centered inertial axes. These curves nearly overlap at orbit scale. The error plot above resolves their differences.

In [ ]:
fig_observer_states = state_histories(result, 'observer')
export(fig_observer_states, ROOT / 'results/gateway_comparison/observer_states.png')

## Gateway position, velocity and acceleration

Compare the true Gateway trajectory with the two tracked trajectories: one uses the perfect observer and one uses the LSO-predicted observer. Their small separation is quantified by the error plot below.

In [ ]:
fig_gateway_states = state_histories(result, 'gateway')
export(fig_gateway_states, ROOT / 'results/gateway_comparison/gateway_states.png')

## 3. Effect of observer uncertainty on Gateway tracking

The uncontrolled halo observer measures Gateway using the same simulated optical observations in both cases. In the first case its true position is known. In the second case its position is estimated using the three LSOs. The two curves therefore show how uncertainty in the observing spacecraft affects Gateway orbit determination.

The relevant tracking errors are

$$
e_{r,G}(t)=\left\|\hat{\mathbf r}_G(t)-\mathbf r_G(t)\right\|,\qquad
e_{v,G}(t)=\left\|\hat{\mathbf v}_G(t)-\mathbf v_G(t)\right\|.
$$

The position and velocity errors below should be compared directly between the two cases. The covariance plot describes the filter's estimated uncertainty; the actual-error plot determines whether tracking accuracy is maintained. Errors are shown in physical Moon-centered inertial coordinates.

In [ ]:
figures = gateway_comparison_figures(result)
fig_errors = figures["errors"]
export(fig_errors, ROOT / "results/gateway_comparison/errors.png")
import matplotlib.pyplot as plt
plt.close(figures["uncertainty"])

In [ ]:
figures_uncertainty = gateway_comparison_figures(result)
plt.close(figures_uncertainty["errors"])
fig_uncertainty = figures_uncertainty["uncertainty"]
export(fig_uncertainty, ROOT / "results/gateway_comparison/uncertainty.png")

## Model and limitations

Propagation uses the Earth–Moon CR3BP. The plots transform normalized rotating states into Moon-centered inertial position, velocity and acceleration. Acceleration is calculated from each state using the dynamics. It is not independently estimated.

The imperfect case retains observer–Gateway cross-covariance. LSO measurements update observer knowledge. Gateway measurements update Gateway without refining the observer. Initial covariances are diagonal, with zero observer–Gateway cross-covariance. The forecast uses zero innovations along the nominal trajectories. The actual estimate uses noisy measurements.

The supplied `SNRHL2` state is a southern NRHO initial condition rounded to four decimals. Its one-period closure miss is about 99 km. This study assumes no stationkeeping or unmodeled forces. Local filter covariance can be overconfident near nonlinear updates, so predicted uncertainty is a conditional comparison rather than a validated error bound.

Gateway's initial state comes from [the earlier cislunar project](https://github.com/scottblender/Gradient-Free-Optimization-for-Cislunar-SDA-2/blob/main/scripts/build_target_case_database.m). Both phases start at model time zero, without a mission epoch. Both trajectories use LU = 384400 km and TU = 375190.25852 s. The source Gateway project used TU = 375695 s.

## 4. Why the uncontrolled halo departs from its reference orbit

The halo observer begins at the rounded SNRHL2 catalog state and is propagated continuously without station-keeping. Because the catalog coordinates are rounded, the resulting trajectory does not close after one period. A nearby initial state is corrected numerically to form the periodic halo reference used for comparison.

The corrected reference returns approximately to the same state every period, but the orbit is **linearly unstable**. Small differences between the uncontrolled spacecraft and the periodic reference can grow with time. The reference-departure metric is

$$
\delta\mathbf{x}_O(t)=\mathbf{x}_{O,\mathrm{uncontrolled}}(t)
-\mathbf{x}_{O,\mathrm{reference}}(t).
$$

The following figures measure how far the uncontrolled spacecraft moves from that periodic orbit. This is not the observer-to-Gateway distance, and it is not Gateway tracking error. The departure matters because it changes the geometry and availability of measurements used for Gateway tracking. The 100 km / 1 m/s observer corridor is deliberately less restrictive than the 100 m / 0.01 m/s Gateway tracking target.

## Stability of the corrected nominal NRHO

This diagnostic evaluates the **corrected periodic nominal orbit**. It is separate from the uncontrolled catalog-seed departure. Integrating the variational equations

$$
\dot{\Phi}(t)=A(t)\Phi(t), \qquad \Phi(0)=I
$$

over one corrected period $T$ gives the **monodromy matrix** $M=\Phi(T)$. Its eigenvalues $\lambda_i$ are **Floquet multipliers**. Their largest magnitude is the **spectral radius**

$$
\rho(M)=\max_i |\lambda_i|.
$$

A multiplier with magnitude greater than one indicates a linearly unstable mode. In that mode, the perturbation amplitude is multiplied by $|\lambda_i|$ after each period at matching orbital phase. A negative real multiplier also reverses the mode's sign each period. This does not mean every initial perturbation or physical error grows by the same factor at every epoch.

For $\rho(M)>1$, the dominant local modal **e-folding time** is

$$
\tau=\frac{T}{\ln\rho(M)}.
$$

It is the timescale for the dominant mode's exponential envelope to grow by a factor of $e$. It is not a maneuver deadline or a prediction of total navigation error. A small **closure error** means the reference returns close to its starting state after one period. It does not establish stability of nearby trajectories.

A second integration checks sensitivity to solver tolerances. The code requires adequate reference closure and a spectral-radius margin before reporting instability. Near-unit results remain unresolved. Unit-circle multipliers alone do not prove nonlinear stability, and repeated unit multipliers require additional care.

The ensemble below starts around the **corrected nominal state**, with seeded 1-km / 0.01-m/s per-axis perturbations and no burns. Its 100-km / 1-m/s corridor is an exploratory requirement independent of the Floquet classification. Non-exits are censored at the study end. Finite-horizon boundedness does not establish nonlinear stability or a quasi-stability classification.

Reference: [NASA's geometrical analysis of station-keeping using Floquet modes](https://ntrs.nasa.gov/citations/20220005226).

In [ ]:
import numpy as np
from halo_tracking.stability import orbit_stability, save_stability
stability = orbit_stability(cfg, periods=5, ensemble_count=8)
save_stability(stability, ROOT / 'results/stability')
stab = stability['summary']
multiplier_rows = ''.join(
    f"<tr><td>{i+1}</td><td>{v['real']:+.6f}</td>"
    f"<td>{v['imag']:+.6f}</td><td>{v['magnitude']:.6f}</td></tr>"
    for i, v in enumerate(stab['floquet_multipliers']))
display(HTML('<table><tr><th>Multiplier</th><th>Real</th>'
             '<th>Imaginary</th><th>Magnitude</th></tr>' + multiplier_rows + '</table>'))
tau = stab['efolding_time_days']
timescale = f'{tau:.2f} days' if tau is not None else 'not established'
display(Markdown(
    f"**Nominal-orbit result: {stab['classification']}.** "
    f"Largest multiplier magnitude: **{stab['spectral_radius']:.6f}**. "
    f"local e-folding time: **{timescale}**. "
    f"The spectral radius changes by {stab['spectral_radius_tolerance_change']:.2e} "
    f"between solver tolerances. One-period position closure is "
    f"{stab['position_closure_km']:.2e} km. "
    "Near-unit neutral multipliers can split numerically. The classification "
    "uses a margin and a closure check, not exact equality to one."
))

fig, axes = plt.subplots(2, 1, figsize=(9.4, 6.2), sharex=True, layout='constrained')
for ax, key, limit, label in zip(
        axes, ['position_departure_km', 'velocity_departure_m_s'],
        [stab['position_limit_km'], stab['velocity_limit_m_s']],
        ['Position departure (km)', 'Velocity departure (m/s)']):
    for i, history in enumerate(stability[key]):
        ax.semilogy(stability['times_tu']/stab['period_tu'],
                    np.maximum(history, 1e-12), color='#245b80', alpha=.45,
                    label='Perturbed uncontrolled arcs' if i == 0 else None)
    ax.axhline(limit, color='#ca7139', ls='--', label='Study corridor limit')
    ax.set_ylabel(label)
    ax.grid(alpha=.17)
axes[-1].set_xlabel('Elapsed corrected nominal periods')
fig.suptitle('Sensitivity to perturbations about the periodic nominal NRHO')
fig.legend(*axes[0].get_legend_handles_labels(), loc='outside lower center',
           ncol=2, frameon=False)
export(fig, ROOT / 'results/stability/nominal_perturbation_departure.png')
display(fig)
plt.close(fig)
display(Markdown(
    f"**Finite-horizon result:** {stab['arcs_with_sampled_corridor_exit']} of "
    f"{stab['ensemble_count']} perturbed arcs leave the study corridor within "
    f"{stab['periods']} nominal periods. Crossings are detected at sampled epochs. "
    "These deviations help expose tracking sensitivity and supply corrective ΔV "
    "cases. They do not establish successful maneuver execution or recovery. "
    "Diagnostics and histories are saved under `results/stability/`."
))

In [ ]:
from halo_tracking.departure import reference_departure, save_departure
from halo_tracking.plotting import departure_figures
departure = reference_departure(cfg, periods=20, ensemble_count=8)
save_departure(departure, cfg, ROOT / 'results/departure')
departure_plots = departure_figures(departure, cfg)
export(departure_plots['departure'], ROOT / 'results/departure/departure.png')
export(departure_plots['period_growth'], ROOT / 'results/departure/period_growth.png')

In [ ]:
closure = departure['diagnostics']
periods = [1, 2, 4, 6, 8, 10, 20]
rows = ''.join(f"<tr><td>{k-1}T → {k}T</td><td>{departure['period_position_rms_km'][k-1]:,.1f}</td>"
               f"<td>{departure['period_velocity_rms_m_s'][k-1]:,.3f}</td></tr>" for k in periods)
display(HTML('<table><tr><th>Period interval</th><th>RMS position departure (km)</th>'
             '<th>RMS velocity departure (m/s)</th></tr>' + rows + '</table>'))
display(Markdown(f"""
The corrected **nominal** one-period closure is **{closure['position_closure_km']:.6f} km** and **{closure['velocity_closure_m_s']:.6f} m/s**. The original rounded **catalog** state's one-period closure error is **{closure['catalog_closure_km']:.1f} km**. The initial rounded truth-to-nominal position offset is **{closure['initial_nominal_offset_km']:.1f} km**.

Each RMS value covers that period alone. Departure can decrease temporarily as the geometry changes. This does not establish stabilization. The guidance section below uses exploratory trigger and corridor values, not operational limits.
"""))

display(Markdown(
    f"**Departure trend:** period RMS grows from "
    f"{departure['period_position_rms_km'][0]:,.1f} km in period 1 to "
    f"{departure['period_position_rms_km'][-1]:,.1f} km in period 20. "
    "The long-horizon motion has left the nominal halo neighborhood. "
    "it illustrates uncontrolled CR3BP divergence, not a maintained orbit."
))


## Cumulative uncontrolled observer trajectories

Each panel extends the same continuous truth propagation: $0\to T$, $0\to2T$, …, $0\to20T$. The dashed curve repeats the corrected nominal orbit at matching phase. Panel annotations give the maximum departure reached up to that horizon.

The displayed grid auto-scales to include the growing trajectory. A second exported grid uses fixed nominal-reference limits to resolve early departure. Later excursions can leave those limits.

In [ ]:
from halo_tracking.plotting import period_trajectory_grid
fig_period_grid = period_trajectory_grid(departure, cfg, scale_mode='local')
export(fig_period_grid, ROOT / 'results/departure/observer_period_grid.png')
display(fig_period_grid)
plt.close(fig_period_grid)

fig_reference_grid = period_trajectory_grid(departure, cfg, scale_mode='reference')
export(fig_reference_grid, ROOT / 'results/departure/observer_period_grid_fixed_scale.png')
plt.close(fig_reference_grid)

## 6. How Gateway tracking degrades without new measurements

The uncontrolled halo observer continues to track Gateway until the Gateway optical measurement link is disabled after eight observer periods. The table below shows how Gateway position-estimation error increases as the estimate is propagated without further Gateway observations. The rolling 24-hour RMSE measures the error over the most recent day instead of averaging across the entire experiment.

The simulated Gateway experiences a small acceleration that is absent from the estimator's CR3BP model. Consequently, its predicted and true trajectories drift apart during the observation gap. The three LSO sites can continue observing the halo spacecraft, but they do not directly measure Gateway. The comparison shows whether better knowledge of the observer is enough to reduce Gateway prediction error once the Gateway optical link is lost.

The **1, 10 and 100 km** entries describe when the simulated Gateway estimation error first exceeds those values. They illustrate the rate at which the uncontrolled tracking solution deteriorates; they are not the eventual **100 m / 0.01 m/s** performance requirements.

In [ ]:
import numpy as np
import os
observer_period_count = int(os.environ.get('HALO_GATEWAY_PERIODS', '20'))

def render_metrics_table(title, headers, records):
    """Compact, horizontally scrollable table for blind prediction results."""
    headings = ''.join(
        '<th style="padding:8px 11px;text-align:right;'
        'background:#eef3f8;border-bottom:1px solid #cbd5e1;">'
        + str(label) + '</th>' for label in headers)
    rows = ''.join(
        '<tr style="background:' + ('#f6f8fc' if i%2 else 'white') + '">'
        + ''.join('<td style="text-align:right;padding:7px 11px;'
                  'border-bottom:1px solid #e5e9f0;white-space:nowrap;">'
                  + str(value) + '</td>' for value in record)
        + '</tr>' for i, record in enumerate(records))
    return (
        '<div style="margin:16px 0 6px;font-weight:600;">' + title +
        '</div><div style="overflow-x:auto;max-height:590px;'
        'border:1px solid #d5dde7;border-radius:8px;">'
        '<table style="border-collapse:collapse;min-width:800px;'
        'font:12px system-ui,sans-serif;font-variant-numeric:tabular-nums;">'
        '<thead><tr>' + headings + '</tr></thead><tbody>' + rows +
        '</tbody></table></div>')

from halo_tracking.gateway_prediction import (
    run_gateway_prediction_study, save_gateway_prediction_study,
    load_gateway_prediction_study
)
from halo_tracking.plotting import gateway_prediction_figure

prediction_periods = observer_period_count
blackout_period = min(int(os.environ.get('HALO_GATEWAY_BLACKOUT_PERIODS', '8')),
                      prediction_periods - 1)
truth_acceleration = float(os.environ.get(
    'HALO_GATEWAY_TRUTH_ACCELERATION_M_S2', '2e-9'))
prediction_output = ROOT / 'results/gateway_prediction'
prediction_kwargs = dict(
    periods=prediction_periods, blackout_period=blackout_period,
    disturbance_m_s2=truth_acceleration, rolling_window_days=1.,
    thresholds_km=(1., 10., 100.))
prediction = load_gateway_prediction_study(
    cfg, prediction_output, **prediction_kwargs)
if prediction is None:
    prediction = run_gateway_prediction_study(cfg, **prediction_kwargs)
    save_gateway_prediction_study(prediction, prediction_output)
    print('Computed new truth-mismatch / forced-blackout prediction scenario.')
else:
    print('Loaded matching saved prediction scenario and time histories.')

fig = gateway_prediction_figure(prediction)
export(fig, prediction_output / 'blind_prediction_endurance.png')
display(fig)
plt.close(fig)

summary = prediction['summary']
display(Markdown(
    f"**Model mismatch:** {summary['disturbance_m_s2']:.2e} m/s² fixed "
    f"inertial direction. **forced blackout starts:** "
    f"{summary['blackout_elapsed_days']:.2f} days. "
    f"**last accepted Gateway observation:** "
    f"{summary['last_accepted_gateway_observation_days']} days. "
    "The estimator uses unforced CR3BP throughout."
))

cols = ['Horizon', 'New Gateway obs.', 'Days since last obs.',
        'Exact obs. error (km)', 'LSO obs. error (km)',
        'Exact obs. 24h RMSE (km)', 'LSO obs. 24h RMSE (km)']
hrows = []
for k in range(1, prediction_periods + 1):
    pair = {r['case']: r for r in prediction['rows']
            if r['horizon_periods'] == k}
    a, b = pair['perfect'], pair['lso_estimated']
    age = a['days_since_gateway_observation']
    hrows.append([f'0 → {k}T', a['gateway_observations_latest_period'],
                  f'{age:.2f}' if np.isfinite(age) else 'No observation yet',
                  f"{a['instantaneous_position_error_km']:.3f}",
                  f"{b['instantaneous_position_error_km']:.3f}",
                  f"{a['rolling_position_rmse_km']:.3f}",
                  f"{b['rolling_position_rmse_km']:.3f}"])
display(HTML(render_metrics_table(
    'Gateway error at each cumulative-horizon endpoint', cols, hrows)))

limits_rows = []
for r in prediction['threshold_rows']:
    value = r['crossing_days_since_last_observation']
    when = (f'{value:.2f}' if value is not None
            else 'Not reached by study end')
    limits_rows.append([r['case'],
                        f"{r['position_threshold_km']:g}", when])
display(HTML(render_metrics_table(
    'First time position-error tolerance is exceeded after blackout',
    ['Observer knowledge', 'Error tolerance (km)',
     'Days after last accepted Gateway observation'], limits_rows)))

final = {r['case']: r for r in prediction['rows']
         if r['horizon_periods'] == prediction_periods}
display(Markdown(
    f"**End-of-study errors:** exact-observer "
    f"{final['perfect']['instantaneous_position_error_km']:,.2f} km. "
    f"LSO-estimated observer "
    f"{final['lso_estimated']['instantaneous_position_error_km']:,.2f} km. "
    "Use the histories and threshold table to judge degradation. "
    "a temporary error decrease does not restart the prediction clock."
))


## 7. Phase-aware station-keeping and NARX dataset

The aim is to predict **when to maneuver and what three-component Δv to apply**, using the estimated observer and Gateway states. The objective is not to force the halo observer to remain within 100 m of its reference. The observer needs only enough orbit control to preserve useful Gateway observations.

| Quantity | Correction is requested when either threshold is exceeded | Reference corridor |
|---|---|---|
| Halo-observer departure | 50 km or 0.5 m/s | 100 km / 1 m/s |
| Gateway orbit-reference departure | 50 m or 0.005 m/s | 100 m / 0.01 m/s |

These are **orbit-correction rules**. The separate mission objective is to keep the **estimated Gateway position error below 100 m** and **velocity error below 0.01 m/s**. The dataset records both types of error, so controller performance can be evaluated against the tracking objective.

### The equations used to calculate maneuvers

The spacecraft states are propagated with the Earth–Moon circular restricted three-body problem (CR3BP):

$$
\ddot{x}-2\dot{y}=\frac{\partial\Omega}{\partial x},\qquad
\ddot{y}+2\dot{x}=\frac{\partial\Omega}{\partial y},\qquad
\ddot{z}=\frac{\partial\Omega}{\partial z},
$$

where $\Omega=(x^2+y^2)/2+(1-\mu)/r_1+\mu/r_2$. An impulsive maneuver changes velocity but not position:

$$
\mathbf{x}(t_b^+)=\mathbf{x}(t_b^-)+
\begin{bmatrix}\mathbf{0}_3\\\Delta\mathbf v\end{bmatrix}.
$$

The STM predicts the first-order effect of a burn on the terminal state. The guidance code solves the regularized weighted least-squares problem

$$
\Delta\mathbf v^*=\arg\min_{\Delta\mathbf v}
\left\|W\left(\mathbf e_f+\Phi_{rv}(t_f,t_b)\Delta\mathbf v\right)\right\|_2^2
+\lambda^2\left\|\Delta\mathbf v\right\|_2^2.
$$

All physical-unit conversions are included in the implementation. Candidate burn phases are compared at the same terminal time, and each candidate is propagated nonlinearly before selection. A smaller terminal error does not necessarily mean either spacecraft has returned inside its corridor.

### What makes the full-loop data useful for NARX

The new closed-loop generator **applies accepted burns to the simulated trajectories**, continues LSO and Gateway optical measurements, and updates the observer/Gateway EKF. Each episode records the resulting navigation estimates, reference deviations, maneuver labels, **actual applied Δv**, burn times and Gateway tracking errors.

A nonlinear ARX model can then use the most recent navigation history and **previous executed maneuvers** to predict the next Δv and burn phase:

$$
[\widehat{\Delta\mathbf v}_k,\widehat{\phi}_k]
=f_{\theta}(\mathbf{u}_k,\ldots,\mathbf{u}_{k-p},
\Delta\mathbf v_{k-1},\ldots,\Delta\mathbf v_{k-q}).
$$

The current maneuver is never provided as an input to its own prediction. Separate simulation episodes are used for training, validation and testing. This is data generation for model identification, **not a claim that a trained NARX controller has achieved the 100 m Gateway tracking requirement**.

In [ ]:
from halo_tracking.identification import generate_dataset
identification_summary = generate_dataset(
    cfg, ROOT / 'results/system_identification')
headers = ['Split', 'Arcs', 'Decisions', 'Triggered',
           'Feasible burns', 'Rejected burns', 'No-burn samples', 'LSO visibility']
rows = ''.join(
    '<tr>' + ''.join(f'<td>{value}</td>' for value in
    [name, item['arcs'], item['decisions'], item['triggered'],
     item['valid_triggered'], item['rejected'], item['no_burn'],
     f"{100*item['visible_fraction']:.1f}%"]) + '</tr>'
    for name, item in identification_summary.items())
display(HTML(
    '<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) +
    '</tr>' + rows + '</table>'))
display(Markdown(
    'Saved **training.npz**, **validation.npz**, **test.npz** and '
    '**metadata.json** to `results/system_identification/`. '
    'Training targets: **target_delta_v_rotating_m_s**. '
    'Always mask examples with **maneuver_label_valid=False**.'
))

triggered = sum(item['triggered'] for item in identification_summary.values())
rejected = sum(item['rejected'] for item in identification_summary.values())
display(Markdown(
    f"**Label coverage:** {triggered - rejected} valid burns and {rejected} rejections "
    "among triggered decisions. Valid labels demonstrate local predicted error reduction, "
    "not closed-loop station-keeping success."
))


In [ ]:
from halo_tracking.narx_dataset import generate_phase_aware_dataset
phase_summary = generate_phase_aware_dataset(cfg, ROOT / 'results/narx_phase_aware')
display(Markdown('**Phase-aware observer and Gateway correction-label coverage** (not a tracking-accuracy result):'))
display(HTML('<table><tr><th>Split</th><th>Arcs</th><th>Decisions</th><th>Valid observer labels</th><th>Valid Gateway labels</th></tr>' + ''.join(
    f"<tr><td>{name}</td><td>{r['arcs']}</td><td>{r['decisions']}</td><td>{r['observer_valid']}</td><td>{r['gateway_valid']}</td></tr>"
    for name, r in phase_summary.items()) + '</table>'))
display(Markdown('These labels are calculated corrections, **not executed maneuvers** or demonstrated 100 m Gateway tracking.'))


In [ ]:
from copy import deepcopy
from halo_tracking.closed_loop_narx import generate_closed_loop_dataset
closed_cfg = deepcopy(cfg)
# A small illustration for notebook execution; the CLI uses the full split sizes.
closed_cfg['narx'].update(training_arcs=1, validation_arcs=1, test_arcs=1)
closed_summary = generate_closed_loop_dataset(closed_cfg, ROOT / 'results/narx_closed_loop_preview')
rows = ''.join(f"<tr><td>{name}</td><td>{item['arcs']}</td><td>{100*item['gateway_position_within_100m_fraction']:.1f}%</td><td>{100*item['gateway_velocity_within_limit_fraction']:.1f}%</td></tr>" for name,item in closed_summary.items())
display(HTML('<table><tr><th>Split</th><th>Arcs</th><th>Gateway position ≤ 100 m</th><th>Gateway velocity ≤ 0.01 m/s</th></tr>'+rows+'</table>'))
display(Markdown('These percentages describe one small simulated sample per split, not validated controller performance.'))
